# Water Potability — Baseline Models

Establishing a first benchmark before any tuning. The goal isn't a good model — it's an honest number to measure future work against.

**Metric: recall.** A false negative means the model calls unsafe water safe and someone drinks it. A false positive just flags a safe sample unnecessarily. The costs aren't symmetric.

**From EDA:** no feature showed meaningful linear correlation with the target (all |r| < 0.05), and the majority class is 61%. Two things to check here — whether linear models fail as that predicts, and whether anything beats the 61% floor.

In [1]:
!pip install xgboost --quiet

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pickle

In [2]:
path = '/kaggle/input/datasets/adityakadiwal/water-potability/water_potability.csv'
df = pd.read_csv(path)

for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df[col].fillna(df[col].median())

print("Missing after imputation:", df.isnull().sum().sum())

X = df.drop('Potability', axis=1)
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Missing after imputation: 0
Train: (2620, 9) Test: (656, 9)


## Preprocessing

Median imputation on `ph`, `Sulfate`, and `Trihalomethanes` (median as a safe default — EDA showed all three near-symmetric, so mean would perform similarly).

80/20 split, stratified to preserve the 61/39 class ratio. 2,620 train / 656 test.

In [3]:
models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(f"{name}: acc={accuracy_score(y_test, pred):.3f}, recall={recall_score(y_test, pred):.3f}")

Dummy: acc=0.610, recall=0.000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression: acc=0.610, recall=0.000
RandomForest: acc=0.659, recall=0.301
XGBoost: acc=0.642, recall=0.398


## Results

| Model | Accuracy | Recall |
|---|---|---|
| Dummy (always predicts 0) | 0.610 | 0.000 |
| LogisticRegression | 0.610 | 0.000 |
| RandomForest | 0.659 | 0.301 |
| XGBoost | **0.642** | **0.398** |

**LogisticRegression matched the dummy exactly.** Same accuracy, zero recall — it learned to always predict "not potable." This confirms the EDA prediction: with no linear relationship between features and target, a linear model has nothing to work with.

**On the convergence warning:** LogisticRegression raised `ConvergenceWarning: lbfgs failed to converge` — it hit the 1,000-iteration limit without settling. The cause is feature scale: `Solids` runs to ~61,000 while `Turbidity` stays under 7, and gradient-based optimizers struggle across that range. Tree models are scale-invariant, so only this one complained.

Scaling the features (`StandardScaler`) would fix the warning, but likely not the result — the model is predicting a constant, which suggests it found no usable signal rather than that it stopped early. Worth verifying rather than assuming: re-run with scaling before concluding linear models are ruled out.

**Tree models beat the floor, but modestly.** RandomForest gains 4.9 accuracy points over the dummy, XGBoost 3.2. Real signal, not much of it.

**Accuracy and recall rank the models oppositely.** RandomForest wins on accuracy; XGBoost wins on recall by nearly 10 points. Since recall is the metric that matters here, **XGBoost is selected as the baseline** — the model that scores worse on the number most people would report.

This is why the metric gets chosen before the models are compared, not after.

## Baseline recorded

**XGBoost, recall = 0.398.** Untuned, no cross-validation, single train/test split. Every future improvement is measured against this.


In [4]:
best = models['XGBoost']
with open('model.pkl', 'wb') as f:
    pickle.dump(best, f)
print("saved")

saved
